# Assignment 1 — Python Data Stack

## Objectives and assessment

Apply array operations, data cleaning, aggregation, joins and visualisation to a small order dataset.
The prerequisite self-check also revisits Python types and mutability.

| Task | Topic | Points |
|---|---|---:|
| 1 | Column-wise standardisation | 2 |
| 2 | In-place modification and independent copies | 2 |
| 3 | Data cleaning and conversion | 3 |
| 4 | Grouped aggregation | 2 |
| 5 | Left joins and row-count validation | 1 |
| 6 | A labelled figure | 1 |
| 7 | Update selected rows with .loc | 2 |
| **Total** | **13 autograded** | **13** |

## Working instructions

1. Run the setup cell before working on the tasks. All input data is supplied in this notebook; no downloads or external files are required.
2. Replace `raise NotImplementedError()` in code answer cells. Preserve function names and signatures and do not edit locked setup or test cells.
3. Return the requested result; printing a value does not return it. Use function parameters rather than hard-coded results or global example data.
4. Restart the kernel, run all cells in order and use **Validate**. Visible checks provide feedback; hidden checks also exercise other inputs satisfying the stated requirements.

## Data and task dependencies

| Object | Contents |
|---|---|
| `M` | Numeric matrix used in tasks 1–2 |
| `RAW` | Orders: `order_id`, `customer_id`, `city`, `amount` (CZK stored as text) |
| `CUSTOMERS` | Customer identifiers and their segment labels |

Tasks 1–2 are independent of the order data. Tasks 4–6 use cleaned orders from task 3;
a failure in `clean` may therefore cause later checks to fail. Functions are also tested with other values, row counts and indices.

## Prerequisite self-check — Types and mutability (not graded)

Predict the values and types after the following Python statements:

```python
raw = "22"
age = int(raw)
original = [age, 38]
alias = original
independent = original.copy()
alias.append(54)
record = ("Braund", original)
record[1][0] = 23
```

1. What are the types of `raw`, `age`, `original` and `record`?
2. What are the final values of `original`, `alias` and `independent`?
3. Which pair has the same identity (`is`), and how does that differ from equality (`==`)?
4. Why can the list inside `record` change even though the tuple is immutable?
5. Does `bool("False")` evaluate to `False`? How is `None` different from an empty string?

<details>
<summary>Self-check explanation</summary>

- Types: `str`, `int`, `list`, `tuple`.
- `original` and `alias` both refer to `[23, 38, 54]`; `independent` remains `[22, 38]`.
- `original is alias` is true. A copied list can have equal values without being the same object.
- A tuple fixes its element references; it does not freeze the mutable list referenced by an element.
- `bool("False")` is true because the string is non-empty. `None` denotes absence, while `""` is a string of length zero.

</details>

## Setup

Run the following cell without modification. The `Agg` backend supports automated grading without a display; task 6 is assessed from the returned `Figure` object.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RAW = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "customer_id": [10, 11, 10, 12, 11, 13, 12, 10],
    "city": [" Prague", "Brno ", "prague", "Brno", " Ostrava ", "Brno", "Prague", "Ostrava"],
    "amount": ["1200.5", "800", "n/a", "450.25", "1100", "", "990.75", "310"],
})

CUSTOMERS = pd.DataFrame({
    "customer_id": [10, 11, 12, 13],
    "segment": ["retail", "business", "retail", "business"],
})

M = np.array([[1.0, 10.0, 100.0],
              [3.0, 20.0, 300.0],
              [5.0, 30.0, 500.0],
              [7.0, 40.0, 700.0]])
print("setup complete")

setup complete


## Task 1 — Column-wise standardisation (2 points)

Implement `zscore_columns(a)`.

| Aspect | Requirement |
|---|---|
| Input | A non-empty two-dimensional numeric NumPy array with finite values and strictly positive standard deviation in each column |
| Calculation | Subtract each column's mean and divide by its population standard deviation (`ddof=0`). |
| Output | A new floating-point NumPy array with the same shape |
| Mutation | Do not modify or share writable storage with the input. |
| Implementation | Use vectorised operations and broadcasting; do not loop over columns. |

For column `j`, each result is `(a[i, j] - mean_j) / std_j`.
Constant columns, missing values and invalid input dimensions are outside the task's scope.
The function must also work with integer arrays and shapes different from `M`.

In [5]:
def zscore_columns(a):
    result = a.copy()
    result = result.astype(float)
    mean = result.mean(axis=0)
    std = result.std(axis=0, ddof=0)
    return (result - mean) / std

In [6]:
z = zscore_columns(M)
assert isinstance(z, np.ndarray) and z.shape == M.shape
np.testing.assert_allclose(z.mean(axis=0), 0, atol=1e-12)
np.testing.assert_allclose(z.std(axis=0, ddof=0), 1, atol=1e-12)

## Task 2 — In-place modification and copies (2 points)

Input to both functions is a writable two-dimensional numeric NumPy array with at least one column.

| Function | Required behaviour | Return value |
|---|---|---|
| `zero_first_column_inplace(a)` | Set the first column of the original array to zero using slicing. Leave other columns unchanged. | `None` |
| `zero_first_column_copy(a)` | Create an independent copy with its first column set to zero. Leave the original unchanged. | The new array |

An independent result must not share memory with the input. Preserve array shape and the values in all other columns.

In [7]:
def zero_first_column_inplace(a):
    a[:, 0] = 0

def zero_first_column_copy(a):
    result = a.copy()
    result[:, 0] = 0
    return result

In [8]:
t = M.copy()
assert zero_first_column_inplace(t) is None
np.testing.assert_allclose(t[:, 0], 0)
np.testing.assert_allclose(t[:, 1:], M[:, 1:])
u = zero_first_column_copy(M)
assert not np.shares_memory(u, M)
np.testing.assert_allclose(u[:, 0], 0)

## Task 3 — Data cleaning (3 points)

Implement `clean(df)` for an order DataFrame with the columns shown in `RAW`.
Input cities are non-missing strings; amounts are strings containing numeric values or invalid entries.
The number of rows and the index may differ from `RAW`; additional columns may be present.

### Required transformation

1. Strip leading and trailing whitespace from `city`; capitalise the first letter and lowercase the remaining letters.
2. Convert `amount` to **floating-point** values, including when all valid strings represent integers. Treat any unparseable value as missing.
3. Remove rows with a missing `amount`, preserving the relative order of the remaining rows.
4. Reset the row index to `0, 1, ..., n-1` without adding an index column.
5. Preserve column order and all other column values in the retained rows.

### Output contract

- Return a new DataFrame; do not modify the input.
- If no valid amounts remain, return an empty frame with the same columns and a floating-point `amount` column.
- For `RAW`, the retained `order_id` values are `[1, 2, 4, 5, 7, 8]`.

**Relevant operations:** `.copy()`, `.str.strip()`, `.str.capitalize()`, `pd.to_numeric`, `.astype(float)`, `.dropna()` and `.reset_index()`.

In [9]:
def clean(df):
    result = df.copy()
    result["city"] = result["city"].str.strip().str.capitalize()
    result["amount"] = pd.to_numeric(result["amount"], errors="coerce").astype( float )
    result = result.dropna(subset=["amount"])
    result = result.reset_index(drop=True)
    return result

In [10]:
c = clean(RAW)
assert isinstance(c, pd.DataFrame)
assert c["order_id"].tolist() == [1, 2, 4, 5, 7, 8]
assert c["amount"].dtype.kind == "f"
assert c.index.tolist() == list(range(6))
assert c["city"].tolist() == ["Prague", "Brno", "Brno", "Ostrava", "Prague", "Ostrava"]

## Task 4 — Grouped aggregation (2 points)

Implement `revenue_by_city(df)` for an already cleaned order DataFrame.

- Return a pandas `Series` indexed by city, with the sum of `amount` for each city.
- Sort totals in descending order; the order of cities with equal totals is unrestricted.
- Do not modify the input. For empty input, return an empty Series.

For `clean(RAW)`, the ordered totals are Prague: `2191.25`, Ostrava: `1410.0`, Brno: `1250.25`.

In [11]:
def revenue_by_city(df):
    result = df.groupby( "city" )["amount"].sum()
    return result.sort_values( ascending=False )

In [12]:
r = revenue_by_city(clean(RAW))
assert isinstance(r, pd.Series)
assert r.index.tolist() == ["Prague", "Ostrava", "Brno"]
np.testing.assert_allclose(r.to_numpy(), [2191.25, 1410.0, 1250.25])

## Task 5 — Left join and row-count validation (1 point)

Implement `attach_segment(orders, customers)`.

| Aspect | Requirement |
|---|---|
| Inputs | Order data containing `customer_id`; customer data containing `customer_id` and `segment`. Keys are non-missing; `orders` does not already contain `segment`. |
| Join | Left-join on `customer_id`, retaining all order columns and adding `segment`. |
| Order | Preserve the original order of the order rows; the output index need not match the input index. |
| Unmatched customer | Retain the order and leave its segment missing. |
| Row multiplication | If the join changes the number of order rows, raise `ValueError("join changed the number of rows")` with this exact message. |
| Mutation | Leave both input frames unchanged. |

A duplicate key in `customers` triggers the error when it matches an order and multiplies rows.
Duplicates with no matching order do not change the row count and do not require an error.

In [26]:
def attach_segment(orders, customers):
    customerSegment = customers[["customer_id","segment"]]
    customerSegmentOrder = orders.merge(customerSegment, on="customer_id", how="left")
    if len(orders) != len(customerSegmentOrder):
        raise ValueError( "join changed the number of rows" )
    return customerSegmentOrder
    
        
    

In [27]:
c = clean(RAW)
j = attach_segment(c, CUSTOMERS)
assert "segment" in j.columns and len(j) == len(c)
assert j["order_id"].tolist() == c["order_id"].tolist()
assert j["segment"].tolist() == ["retail", "business", "retail", "business", "retail", "retail"]

## Task 6 — Data visualisation (1 point)

Implement `make_figure(df)` for a non-empty cleaned order DataFrame with non-missing cities and positive amounts in CZK.
Return a `matplotlib.figure.Figure` with exactly two axes arranged in one row.

| Element | Requirement |
|---|---|
| Left axes | Histogram of `amount` showing order counts, with at least two bins |
| Right axes | Vertical bar chart of total revenue by city, with one bar per city |
| Labels | Non-empty x and y labels on both axes; use meaningful quantities and units where applicable. |
| Figure title | A non-empty overall title set with `suptitle` |

Bin count, colours, figure size, exact label wording and city order are otherwise unrestricted.
Return the figure rather than the axes or the result of `plt.show()`; do not modify the input.
The function must derive the plotted data from its parameter, not from `RAW`.

In [30]:
def make_figure( df ):
    figure, axes = plt.subplots( 1, 2 )
    axes[0].hist( df["amount"], bins=5 )
    axes[0].set_xlabel("Amount")
    axes[0].set_ylabel("Orders")
    revenue = df.groupby("city")["amount"].sum()
    axes[1].set_xlabel("City")
    axes[1].set_ylabel("Revenue")
    axes[1].bar(revenue.index, revenue.values)
    figure.suptitle("Orders and revenue")
    return figure

In [31]:
f = make_figure(clean(RAW))
assert isinstance(f, matplotlib.figure.Figure)
assert len(f.axes) == 2
for ax in f.axes:
    assert ax.get_xlabel().strip() and ax.get_ylabel().strip()
assert f._suptitle is not None and f._suptitle.get_text().strip()

## Task 7 — Update selected rows (2 points)

Implement `set_positive_rows(df)`. The input is a pandas DataFrame with numeric columns
`a` and `b` and no missing values; it may contain other columns and may be empty.
Modify the original DataFrame: set `b` to `1` exactly where `a > 0`, leaving all other
values, columns and index labels unchanged. Return `None`. Use a single `.loc` assignment,
not chained indexing. Tests include non-default and duplicate index labels.

In [32]:
def set_positive_rows(df):
    df.loc[df["a"] > 0, "b"] = 1

In [33]:
_df = pd.DataFrame({"a": [-1, 0, 2], "b": [5, 6, 7]})
assert set_positive_rows(_df) is None
assert _df["b"].tolist() == [5, 6, 1]